In [ ]:
import pdfplumber as pp

pdf_path="MLBB_Match_Stats_Games_1_to_20_Uncompressed.pdf"

with pp.open(pdf_path) as pdf:
   print("pages:",len(pdf.pages)) 

   first_page = pdf.pages[0]
   text = first_page.extract_text()

   print(text[:2000])

starting the project 

In [1]:
import pdfplumber as pp

pdf_path="MLBB_Match_Stats_Games_1_to_20_Uncompressed.pdf"

all_text=""

with pp.open(pdf_path) as pdf:
    for page in pdf.pages:
        all_text += page.extract_text() + "\n"


print(all_text[:5000])

with open("mlbb_raw.text","w", encoding="utf-8") as f:
    f.write(all_text)

print('saved!')

MLBB Live Win Probability Dataset
Time-Series Features | Uncompressed Multi-Game Aggregation (Games 1 - 20)
GAME TIMESTAMP KILLS (T1) KILLS (T2) GOLD (T1) GOLD (T2) GOLD DIFF TURRETS (T1) TURRETS (T2) TURTLES (T1) TURTLES (T2) LORDS (T1) LORDS (T2) WINNER
03:01 2 0 8.2K 7.2K T1 +985 1 0 0 0 0 0 -
05:02 3 0 13.1K 12.2K T1 +963 1 0 0 1 0 0 -
10:01 8 1 28.6K 25.8K T1 +2.8K 2 2 2 1 0 0 -
1
15:01 12 1 45.4K 37.1K T1 +8.2K 8 2 2 1 2 0 -
20:01 17 6 57.9K 55.8K T1 +2.1K 8 6 2 1 3 0 -
21:17 (End) 22 6 65.9K 59.0K T1 +6.8K 9 6 2 1 4 0 Team 1
03:01 1 2 7.3K 8.2K T2 +961 0 0 0 0 0 0 -
05:00 2 5 12.3K 14.1K T2 +1.7K 0 1 0 1 0 0 -
2
10:00 3 12 24.1K 32.7K T2 +8.7K 0 4 0 2 0 0 -
14:21 (End) 4 17 35.3K 49.5K T2 +14.2K 0 9 1 2 0 2 Team 2
03:00 2 1 8.2K 7.4K T1 +802 0 0 0 0 0 0 -
05:00 4 3 13.7K 13.1K T1 +642 0 0 0 1 0 0 -
3
10:00 7 6 28.5K 27.5K T1 +972 2 2 1 1 0 0 -
14:31 (End) 10 15 41.0K 47.9K T2 +6.8K 7 7 1 2 1 1 Team 2
03:00 0 1 7.5K 7.4K T1 +138 0 0 0 0 0 0 -
05:00 0 2 12.7K 12.7K 0 0 0 0 1 0 0 -

inspection

In [ ]:
lines = all_text.split("\n")

for i, line in enumerate(lines[:20]):
    print(i,":",line)

#print(lines)


0 : MLBB Live Win Probability Dataset
1 : Time-Series Features | Uncompressed Multi-Game Aggregation (Games 1 - 20)
2 : GAME TIMESTAMP KILLS (T1) KILLS (T2) GOLD (T1) GOLD (T2) GOLD DIFF TURRETS (T1) TURRETS (T2) TURTLES (T1) TURTLES (T2) LORDS (T1) LORDS (T2) WINNER
3 : 03:01 2 0 8.2K 7.2K T1 +985 1 0 0 0 0 0 -
4 : 05:02 3 0 13.1K 12.2K T1 +963 1 0 0 1 0 0 -
5 : 10:01 8 1 28.6K 25.8K T1 +2.8K 2 2 2 1 0 0 -
6 : 1
7 : 15:01 12 1 45.4K 37.1K T1 +8.2K 8 2 2 1 2 0 -
8 : 20:01 17 6 57.9K 55.8K T1 +2.1K 8 6 2 1 3 0 -
9 : 21:17 (End) 22 6 65.9K 59.0K T1 +6.8K 9 6 2 1 4 0 Team 1
10 : 03:01 1 2 7.3K 8.2K T2 +961 0 0 0 0 0 0 -
11 : 05:00 2 5 12.3K 14.1K T2 +1.7K 0 1 0 1 0 0 -
12 : 2
13 : 10:00 3 12 24.1K 32.7K T2 +8.7K 0 4 0 2 0 0 -
14 : 14:21 (End) 4 17 35.3K 49.5K T2 +14.2K 0 9 1 2 0 2 Team 2
15 : 03:00 2 1 8.2K 7.4K T1 +802 0 0 0 0 0 0 -
16 : 05:00 4 3 13.7K 13.1K T1 +642 0 0 0 1 0 0 -
17 : 3
18 : 10:00 7 6 28.5K 27.5K T1 +972 2 2 1 1 0 0 -
19 : 14:31 (End) 10 15 41.0K 47.9K T2 +6.8K 7 7 1 2 

In [ ]:
sample = lines[3]
tokens = sample.split()
print(tokens)
print(sample)

['03:01', '2', '0', '8.2K', '7.2K', 'T1', '+985', '1', '0', '0', '0', '0', '0', '-']
03:01 2 0 8.2K 7.2K T1 +985 1 0 0 0 0 0 -


In [ ]:
def convert_gold(value):
    
    value = value.replace("+", "")
    
    if "K" in value:
        return float(value.replace("K", "")) * 1000
    
    return float(value)


print(convert_gold("8.2K"))
print(convert_gold("+2.8K"))
print(convert_gold("+985"))

8200.0
2800.0
985.0


In [ ]:
def parse_line(tokens):

    # ---------- CASE 1 : normal row (14 tokens) ----------
    if len(tokens) == 14:

        minute = int(tokens[0].split(":")[0])

        kills_t1 = int(tokens[1])
        kills_t2 = int(tokens[2])

        gold_t1 = convert_gold(tokens[3])
        gold_t2 = convert_gold(tokens[4])

        turrets_t1 = int(tokens[7])
        turrets_t2 = int(tokens[8])

        turtles_t1 = int(tokens[9])
        turtles_t2 = int(tokens[10])

        lords_t1 = int(tokens[11])
        lords_t2 = int(tokens[12])



    # ---------- CASE 2 : equal gold row (13 tokens) ----------
    elif len(tokens) == 13:

        minute = int(tokens[0].split(":")[0])

        kills_t1 = int(tokens[1])
        kills_t2 = int(tokens[2])

        gold_t1 = convert_gold(tokens[3])
        gold_t2 = convert_gold(tokens[4])

        # indexes shifted because no T1/T2 and no +gold diff
        turrets_t1 = int(tokens[5])
        turrets_t2 = int(tokens[6])

        turtles_t1 = int(tokens[7])
        turtles_t2 = int(tokens[8])

        lords_t1 = int(tokens[9])
        lords_t2 = int(tokens[10])



    return {
        "minute": minute,
        "kill_diff": kills_t1 - kills_t2,
        "gold_diff": gold_t1 - gold_t2,
        "turret_diff": turrets_t1 - turrets_t2,
        "turtle_diff": turtles_t1 - turtles_t2,
        "lord_diff": lords_t1 - lords_t2
    }

In [ ]:
for line in lines[:20]:
    
    # case 1 : game number
    if line.isdigit():
        print("GAME ID --->", line)
        
    # case 2 : end row
    elif "(End)" in line:
        print("END ROW --->", line)
        
    # case 3 : normal row
    elif ":" in line:
        print("NORMAL --->", line)
        
    else:
        print("IGNORE --->", line)

IGNORE ---> MLBB Live Win Probability Dataset
IGNORE ---> Time-Series Features | Uncompressed Multi-Game Aggregation (Games 1 - 20)
IGNORE ---> GAME TIMESTAMP KILLS (T1) KILLS (T2) GOLD (T1) GOLD (T2) GOLD DIFF TURRETS (T1) TURRETS (T2) TURTLES (T1) TURTLES (T2) LORDS (T1) LORDS (T2) WINNER
NORMAL ---> 03:01 2 0 8.2K 7.2K T1 +985 1 0 0 0 0 0 -
NORMAL ---> 05:02 3 0 13.1K 12.2K T1 +963 1 0 0 1 0 0 -
NORMAL ---> 10:01 8 1 28.6K 25.8K T1 +2.8K 2 2 2 1 0 0 -
GAME ID ---> 1
NORMAL ---> 15:01 12 1 45.4K 37.1K T1 +8.2K 8 2 2 1 2 0 -
NORMAL ---> 20:01 17 6 57.9K 55.8K T1 +2.1K 8 6 2 1 3 0 -
END ROW ---> 21:17 (End) 22 6 65.9K 59.0K T1 +6.8K 9 6 2 1 4 0 Team 1
NORMAL ---> 03:01 1 2 7.3K 8.2K T2 +961 0 0 0 0 0 0 -
NORMAL ---> 05:00 2 5 12.3K 14.1K T2 +1.7K 0 1 0 1 0 0 -
GAME ID ---> 2
NORMAL ---> 10:00 3 12 24.1K 32.7K T2 +8.7K 0 4 0 2 0 0 -
END ROW ---> 14:21 (End) 4 17 35.3K 49.5K T2 +14.2K 0 9 1 2 0 2 Team 2
NORMAL ---> 03:00 2 1 8.2K 7.4K T1 +802 0 0 0 0 0 0 -
NORMAL ---> 05:00 4 3 13.7K 13.

In [ ]:
for line in lines:

    if ":" in line and "(End)" not in line:

        tokens = line.split()

        if len(tokens) != 14:
            print(len(tokens), "--->", tokens)

for line in lines:

    if ":" in line and "(End)" not in line:

        tokens = line.split()

        # check if first token is game id accidentally attached
        if len(tokens) == 15:
            print("BROKEN GAME ID:", tokens)

13 ---> ['05:00', '0', '2', '12.7K', '12.7K', '0', '0', '0', '0', '1', '0', '0', '-']
15 ---> ['6', '10:00', '14', '3', '30.2K', '26.5K', 'T1', '+3.6K', '2', '0', '2', '0', '0', '0', '-']
15 ---> ['7', '02:55', '0', '4', '7.1K', '8.3K', 'T2', '+1.2K', '0', '0', '0', '0', '0', '0', '-']
15 ---> ['12', '15:00', '12', '12', '47.4K', '47.9K', 'T2', '+437', '3', '0', '2', '0', '0', '0', '-']
15 ---> ['15', '03:00', '1', '0', '7.9K', '7.3K', 'T1', '+606', '0', '0', '0', '0', '0', '0', '-']
15 ---> ['18', '10:00', '3', '1', '33.6K', '23.2K', 'T1', '+10.4K', '5', '0', '2', '0', '0', '0', '-']
BROKEN GAME ID: ['6', '10:00', '14', '3', '30.2K', '26.5K', 'T1', '+3.6K', '2', '0', '2', '0', '0', '0', '-']
BROKEN GAME ID: ['7', '02:55', '0', '4', '7.1K', '8.3K', 'T2', '+1.2K', '0', '0', '0', '0', '0', '0', '-']
BROKEN GAME ID: ['12', '15:00', '12', '12', '47.4K', '47.9K', 'T2', '+437', '3', '0', '2', '0', '0', '0', '-']
BROKEN GAME ID: ['15', '03:00', '1', '0', '7.9K', '7.3K', 'T1', '+606', '0', '0'

In [ ]:
for line in lines:

    if ":" in line and "(End)" not in line:

        tokens = line.split()

        if len(tokens) == 15:
            print(tokens)

In [ ]:
all_rows = []

current_game_rows = []
current_game_id = None


for line in lines:

    # ---------- CASE 1 : GAME ID ----------
    if line.isdigit():

        current_game_id = int(line)


    # ---------- CASE 2 : NORMAL ROW ----------
    elif ":" in line and "(End)" not in line:

        tokens = line.split()

        if len(tokens) == 15:
            current_game_id = int(tokens[0])
            tokens = tokens[1:]

        row = parse_line(tokens)

        current_game_rows.append(row)


    # ---------- CASE 3 : END ROW ----------
    elif "(End)" in line:

        # detect winner
        if "Team 1" in line:
            winner = 1
        else:
            winner = 0


        # add winner + game id to every row of this match
        for row in current_game_rows:

            row["game_id"] = current_game_id
            row["winner"] = winner

            all_rows.append(row)


        # clear temporary storage for next game
        current_game_rows = []

print(all_rows)

[{'minute': 3, 'kill_diff': 2, 'gold_diff': 1000.0, 'turret_diff': 1, 'turtle_diff': 0, 'lord_diff': 0, 'game_id': 1, 'winner': 1}, {'minute': 5, 'kill_diff': 3, 'gold_diff': 900.0, 'turret_diff': 1, 'turtle_diff': -1, 'lord_diff': 0, 'game_id': 1, 'winner': 1}, {'minute': 10, 'kill_diff': 7, 'gold_diff': 2800.0, 'turret_diff': 0, 'turtle_diff': 1, 'lord_diff': 0, 'game_id': 1, 'winner': 1}, {'minute': 15, 'kill_diff': 11, 'gold_diff': 8300.0, 'turret_diff': 6, 'turtle_diff': 1, 'lord_diff': 2, 'game_id': 1, 'winner': 1}, {'minute': 20, 'kill_diff': 11, 'gold_diff': 2100.0, 'turret_diff': 2, 'turtle_diff': 1, 'lord_diff': 3, 'game_id': 1, 'winner': 1}, {'minute': 3, 'kill_diff': -1, 'gold_diff': -900.0, 'turret_diff': 0, 'turtle_diff': 0, 'lord_diff': 0, 'game_id': 2, 'winner': 0}, {'minute': 5, 'kill_diff': -3, 'gold_diff': -1800.0, 'turret_diff': -1, 'turtle_diff': -1, 'lord_diff': 0, 'game_id': 2, 'winner': 0}, {'minute': 10, 'kill_diff': -9, 'gold_diff': -8600.000000000004, 'turret

In [ ]:
import pandas as pd

df = pd.DataFrame(all_rows)

df["gold_diff"] = df["gold_diff"].round(2)

df.to_csv("mlbb_dataset.csv", index=False)

print(df.head())

   minute  kill_diff  gold_diff  turret_diff  turtle_diff  lord_diff  game_id  \
0       3          2     1000.0            1            0          0        1   
1       5          3      900.0            1           -1          0        1   
2      10          7     2800.0            0            1          0        1   
3      15         11     8300.0            6            1          2        1   
4      20         11     2100.0            2            1          3        1   

   winner  
0       1  
1       1  
2       1  
3       1  
4       1  


In [ ]:
print(df.shape)
print(df.describe())
print(df["winner"].value_counts())

(75, 8)
          minute  kill_diff     gold_diff  turret_diff  turtle_diff  \
count  75.000000  75.000000     75.000000    75.000000     75.00000   
mean    8.253333  -0.226667    217.333333     0.053333      0.12000   
std     5.472519   5.371530   4019.223178     2.598822      1.22981   
min     2.000000 -15.000000 -13700.000000    -7.000000     -3.00000   
25%     3.000000  -3.000000  -1250.000000     0.000000     -1.00000   
50%     5.000000   1.000000    100.000000     0.000000      0.00000   
75%    10.000000   2.000000   1750.000000     1.000000      1.00000   
max    25.000000  14.000000  11300.000000     9.000000      3.00000   

       lord_diff    game_id    winner  
count  75.000000  75.000000  75.00000  
mean    0.080000  10.426667   0.56000  
std     0.652645   5.877519   0.49973  
min    -2.000000   1.000000   0.00000  
25%     0.000000   5.500000   0.00000  
50%     0.000000  10.000000   1.00000  
75%     0.000000  15.000000   1.00000  
max     3.000000  20.000000   1.